<a href="https://colab.research.google.com/github/sethmwhite/worth-it/blob/main/Worth_It.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WorthIt
My project helps gamers on a budget decide which Steam games to buy this week by matching highly rated games to the genre, budget, and operating system they choose, as well as checking if a game is on sale.

#Setup:
Imports all the tools we need (pandas for data, requests for the CheapShark API) and also loads 'steam_games.csv' in 'games'

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import requests
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

#Fit:
Which highly rated games match the genre and budget the gamer types in?

*   'fit_games' keeps games in the chosen genre that costs no more than the given budget, with the option to leave free games. Sorted best rated first.



In [ ]:
#Return the games in a genre that cost no more than budget, best rated first.

#Set include_free=False, or change with_free to no_free (in the print) to leave out free games.


def fit_games(games, genre, budget, include_free=True):
    in_genre = games[games["genres"].str.contains(genre, case=False, na=False)]
    affordable = in_genre[in_genre["price"] <= budget]
    if include_free == False:
        affordable = affordable[affordable["price"] > 0]
    return affordable.sort_values("pct_pos_total", ascending=False)

with_free = fit_games(games, "Action", 30, include_free=True)
no_free = fit_games(games, "Action", 40, include_free=False)

print(with_free[["name", "price", "pct_pos_total"]].head(10))

#Compatibility:
Which of them run on the gamer’s operating system (Windows, Mac, Linux, etc.)?

*   Return the games that run on the given os_name ('windows', 'mac', or 'linux').

* Prints the number of "fitting" games based on your search, as well as the number of those games that run on your OS.








In [ ]:
def games_for_os(games, os_name):
    return games[games[os_name] == True]

fit = fit_games(games, "RPG", 60, include_free=False)
runs_on_mac = games_for_os(fit, "mac")

print(f"Matching Games: {len(fit)}")
print(f"OS-Compatible Games: {len(runs_on_mac)}")
print(runs_on_mac[["name", "price", "pct_pos_total"]].head(10))

#Deals (CheapShark AI)
Which of them are on sale today, and how close is it to its lowest price ever (CheapShark)?

*   'get_lowest_price' asks the CheapShark API for a game's lowest price ever recorded, using the game's Steam 'appid'



In [ ]:
#Return the current price and the historically lowest price (based on CheapShark's values) on the highest rated games.

def get_lowest_price(appid):
    url = "https://www.cheapshark.com/api/1.0/games"
    headers = {"User-Agent": "WorthIt/1.0 (MIST 4600 class project)"}
    response = requests.get(url, params={"steamAppID": appid}, headers=headers)
    if response.status_code != 200:
        return None
    data = response.json()
    if len(data) == 0:
        return None
    return float(data[0]["cheapest"])

In [ ]:
test = runs_on_mac.head(15)
for i in range(len(test)):
    row = test.iloc[i]
    print(f"{row['name']} - Current: ${row['price']:.2f} Lowest: ${get_lowest_price(row['appid'])}")

#Class: GamePick
The 'GamePick' has everything the app knows about a recommended game - its name, current price, lowest recorded price ever (from CheapShark), and the percent of positive reviews.

* max_discount() calculates how far below its current price the game has sold
* card() writes a one-line summarization of the game's details.

In [ ]:
class GamePick:
    def __init__(self, name, price, rating, lowest):
        self.name = name
        self.price = price
        self.rating = rating
        self.lowest = lowest

    def max_discount(self):
        if self.lowest is None or self.price == 0:
          return 0
        return round ((self.price - self.lowest) / self.price *100)

    def card(self):
      if self.lowest is None:
          return f"{self.name}: ${self.price:.2f}, {self.rating:.0f}% positive, lowest price not available."
      return f"{self.name}: ${self.price:.2f}, {self.rating:.0f}% positive, has sold for {self.max_discount()}% off (Lowest - ${self.lowest:.2f})."

##Final Recommendation
The user's choices (genre, budget, OS, and whether or not they include free games) are at the top of the cell.
The app finds the games that fit (Sub-question 1), keeps the ones that run on the user's OS (Sub-question 2), looks for each games' lowest price recorded on CheapShark (Sub-question 3), and prints a short recommendation for each.

In [ ]:
# The gamer's choices: change these to try other searches
genre = "Action"
budget = 30
os_name = "mac"
include_free = False

fit = fit_games(games, genre, budget, include_free)
compatible = games_for_os(fit, os_name)
shortlist = compatible.head(10)

picks = []
for i in range(len(shortlist)):
    row = shortlist.iloc[i]
    lowest = get_lowest_price(row["appid"])
    pick = GamePick(row["name"], row["price"], row["pct_pos_total"], lowest)
    picks.append(pick)

print(f"Games that fit the genre and budget: {len(fit)}")
print(f"Of those, games that run on {os_name}: {len(compatible)}")
print()
print("WorthIt picks, best rated first:")
for pick in picks:
    print(pick.card())